In [ ]:
from hybrid_joint import *
from diffrax import *
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from controls import *

Same equation as in `dynamics_rough_vol.ipynb`, but with `M` trajectories.

In [ ]:
# global parameters

H = 0.1
M = 1000
grid_points = 1000000
kappa = 1
T = 1
rho = 0.8
fine_times = jnp.arange(0, 1, 1/grid_points)

In [ ]:
# Don't run! 30k sample paths take > 1 minute on my laptop

#hybrid_scheme(grid_points, 30000, T, H, kappa)

In [ ]:
# Don't run! 3 x 10k sample paths take 41 seconds

#for i in range(10000):
#    fbm, bm1, bm2 = correlated_fbm_bm_bm(0.8, 0, 0, grid_points, T, 0.1, kappa)

In [ ]:
# Let's parallelise the creation of the iid noise paths
# See the file hybrid_joint for the options regarding different correlation structures. Different functions are needed for fully correlated noise

from concurrent.futures import ProcessPoolExecutor

num_processes = 8  # Adjust based on your machine's CPU cores and task requirements

args = (rho, grid_points, T, H, kappa)

with ProcessPoolExecutor(max_workers=num_processes) as executor:
    futures = [executor.submit(correlated_fbm_bm_wrapper, args) for _ in range(M)]
    results = [future.result() for future in futures]

# 3 x 10k sample paths takes 10 seconds on my laptop, much better

# keep in mind though, unless the rho's are parameters this only needs to be run once for all passes of the calibration (unless there is something to be
# gained from using independent noises across different calibration steps)

# it's worth checking that the different sample paths are indeed independent

list_fbm, list_bm = zip(*results)

stacked_fbm = jnp.stack(list_fbm).T
stacked_bm = jnp.stack(list_bm).T

In [ ]:
#np.savez('error_data.npz', stacked_fbm = stacked_fbm, stacked_bm = stacked_bm)

In [ ]:
loaded = np.load('error_data.npz')
stacked_fbm = jnp.array(loaded['stacked_fbm'])
stacked_bm = jnp.array(loaded['stacked_bm'])

In [ ]:
sample = 100

plt.figure(figsize=(20, 5))
plt.plot(fine_times, stacked_fbm[:,sample], color = 'red')
plt.plot(fine_times, stacked_bm[:,sample], color = 'green')

In [ ]:
epsilon = 0.000001
lag = 1.2*epsilon
solver_epsilon = 0.1*epsilon
times = jnp.arange(0, 1, epsilon)
print(times.shape)
save_times = jnp.arange(0, 1, 0.01)
every_nth_pt = int(grid_points*epsilon)
print(every_nth_pt)

subsampled_bm = stacked_bm[::every_nth_pt]
bm_interp = LinearInterpolation(times, subsampled_bm)
bm_control = evaluate_and_reshape(bm_interp.evaluate)

subsampled_fbm = stacked_fbm[::every_nth_pt]
fbm_interp = LinearInterpolation(times, subsampled_fbm)
fbm_fun = evaluate_and_reshape_fun(fbm_interp.evaluate)
fbm_lagged_control = lagged_control(fbm_fun, lag)

lead_lag_control = stack_controls(fbm_lagged_control, bm_control)

In [ ]:
# define the SDE
# these parameters could be put inside the args of drift and diffusion
sigma0 = 0.1
sigma1 = 0.1

gamma0 = 1
gamma1 = 1

alpha = 0.1
beta = 0.1

a = 1
b = 0.1
c = 0.1

saveat = SaveAt(ts = times)

# the drift, diffusion and initial condition of the RDE. This can be changed to any other RDE. In particular, if one wants the vol to be
# driven by a single noise, just set the corresponding term in the vector field to zero
y0 = jnp.array([1, 1])
drift = lambda t, y, args: jnp.array([- (1/2) * sigma1 * a * y[0] * jnp.power(a*((y[1] - b)**2) + c, -1/2) * (y[1] - b) * jnp.power(y[1],gamma1) - (1/2) * (a*((y[1] - b)**2) + c) * y[0], alpha + beta * y[1]])
diffusion = lambda t, y, args: jnp.array([[0, jnp.sqrt(a*((y[1] - b)**2) + c) * y[0]], [sigma0 * jnp.power(y[1],gamma0), sigma1 * jnp.power(y[1],gamma1)]])

In [ ]:
solutions = vmap_batch_fbm_solve(jnp.arange(M), lead_lag_control, drift, diffusion, y0, Midpoint(), 0, 1, solver_epsilon, saveat)

In [ ]:
np.savez('solution.npz', sol = solutions.ys)

In [ ]:
loaded = np.load('solution.npz')
sol = jnp.array(loaded['sol'])

In [ ]:
solutions_coarse = vmap_batch_fbm_solve(jnp.arange(M), lead_lag_control, drift, diffusion, y0, Midpoint(), 0, 1, solver_epsilon, saveat)

In [ ]:
i = 1

sol_incr = solutions.ys[:, 0, i] - jnp.ones_like(solutions.ys[:, 0, i])

exclude = 3
perc = (M - exclude)*100/M
M_excl = M - exclude

diff = solutions.ys[:, 0, i] - solutions_coarse.ys[:, 0, i]

threshold = jnp.percentile(jnp.abs(diff), perc)
indices_to_remove = jnp.where(jnp.abs(diff) >= threshold)[0]
diff_filtered = diff[jnp.abs(diff) < threshold]
sol_incr_filtered = jnp.delete(sol_incr, indices_to_remove)

def print_errors():
    print(jnp.linalg.norm(diff)/jnp.sqrt(M))
    print(jnp.linalg.norm(sol_incr)/jnp.sqrt(M))
    print((jnp.linalg.norm(diff)/jnp.sqrt(M))/(jnp.linalg.norm(sol_incr)/jnp.sqrt(M))) 

def print_errors_filtered():
    print(jnp.linalg.norm(diff_filtered)/jnp.sqrt(M_excl))
    print(jnp.linalg.norm(sol_incr_filtered)/jnp.sqrt(M_excl))
    print((jnp.linalg.norm(diff_filtered)/jnp.sqrt(M_excl))/(jnp.linalg.norm(sol_incr_filtered)/jnp.sqrt(M_excl))) 
    
print_errors()

In [ ]:
# Assuming solutions.ys is a JAX array
data = diff

# Plot the data with dots
plt.plot(data, 'o')
plt.show()

In [ ]:
vol = sol[:, :, 1].T
Z = LinearInterpolation(times, vol).evaluate #create the vol process

In [ ]:
y0 = jnp.zeros(M)

drift_exp = lambda t, y, args: jnp.array(-(1/2) * ( a * jnp.square(Z(t) - b) + c ))
diffusion_exp = lambda t, y, args: jnp.diag(jnp.array(jnp.sqrt(a * jnp.square(Z(t) - b) + c)))
terms_exp = MultiTerm(ODETerm(drift_exp), ControlTerm(diffusion_exp, bm_control))

integral_price = diffeqsolve(terms_exp, Euler(), 0, 1, dt0 = epsilon, max_steps=None, y0=y0, saveat=saveat) # note solver has to have size epsilon for this to be Ito
print(integral_price.ys.shape)
exp_martingale = jnp.exp(integral_price.ys)

In [ ]:
k = 103
plt.figure(figsize=(20, 5))
plt.plot(times, solutions_coarse.ys[k, :, 0])
plt.plot(times, exp_martingale[:,k])
print(exp_martingale[-1, k] - solutions_coarse.ys[k, -1, 0])

In [ ]:
diff_exp = exp_martingale[-1, :] - solutions_coarse.ys[:, -1, 0]
sol_incr_exp = solutions_coarse.ys[:, -1, 0] - solutions_coarse.ys[:, 0, 0]
print((jnp.linalg.norm(diff_exp)/jnp.sqrt(M))/(jnp.linalg.norm(sol_incr_exp)/jnp.sqrt(M)))

In [ ]:
exclude = 3
perc = (M - exclude)*100/M
M_excl = M - exclude

threshold = jnp.percentile(jnp.abs(diff_exp), perc)
indices_to_remove = jnp.where(jnp.abs(diff_exp) >= threshold)[0]
diff_filtered_exp = diff_exp[jnp.abs(diff_exp) < threshold]
sol_incr_exp_filtered = jnp.delete(sol_incr_exp, indices_to_remove)
print(diff_filtered_exp.shape, sol_incr_filtered.shape)

print((jnp.linalg.norm(diff_filtered)/jnp.sqrt(M_excl))/(jnp.linalg.norm(sol_incr_exp_filtered)/jnp.sqrt(M_excl)))

In [ ]:
plt.plot(diff_exp, 'o')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import jax.numpy as jnp

# Example index k
k = 3  # Replace with your desired index

# Plot real_sol[k, :] and exp_martingale[k, :]
plt.figure(figsize=(10, 5))

plt.plot(real_sol[k, :])
plt.plot(exp_martingale[k, :])

plt.show()